In [29]:
import pandas as pd
import numpy as np


base_file_path = "../../data/raw/"
save_file_path = "../../data/iterim/"


def convert_to_integer(column: pd.Series):
    converted_column = column.astype("string").str.replace(" ", "").str.replace(",", ".").str.replace("\u00a0", "", regex=False).str.replace("—", "-", regex=False)
    converted_colum = pd.to_numeric(converted_column , errors="coerce")
    return converted_colum.round(2)


def convert_to_datetime(column: pd.Series):
    converted_column = pd.to_datetime(column, errors="coerce", format="mixed")
    return converted_column


def convert_to_string(column: pd.Series):
    converted_column = column.astype("string").str.strip().str.replace(" ", "").str.lower()
    return converted_column

In [30]:
head_df = pd.read_csv(f"{base_file_path}head.csv")

In [47]:
display(head_df.describe())
display(head_df.info())
display(head_df)

,head,t
count,4525.0,4525
mean,38.802519,2023-01-04 05:57:03.381215
min,25.75,2020-01-03 00:00:00
25%,33.57,2021-07-14 00:00:00
50%,40.6,2023-01-16 00:00:00
75%,43.74,2024-07-04 00:00:00
max,48.56,2025-12-25 00:00:00
std,5.958204,NaN


<class 'pandas.DataFrame'>
Index: 4525 entries, 0 to 4549
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   well_id    4525 non-null   str           
 1   head       4525 non-null   Float64       
 2   t          4525 non-null   datetime64[us]
 3   record_id  4525 non-null   str           
 4   unit       4525 non-null   string        
 5   quality    4525 non-null   string        
dtypes: Float64(1), datetime64[us](1), str(2), string(2)
memory usage: 462.6 KB


None

,well_id,head,t,record_id,unit,quality
0,BH-134,36.65,2025-12-23,HD-000636,m,ok
1,BH-084,45.67,2020-02-25,HD-000738,m,ok
2,BH-089,40.76,2020-01-16,HD-002657,m,ok
3,BH-083,42.24,2025-01-18,HD-002712,m,ok
4,BH-067,41.64,2022-01-18,HD-001171,m,ok
...,...,...,...,...,...,...
4545,BH-240,31.19,2021-08-20,HD-001560,m,ok
4546,BH-156,40.92,2021-04-13,HD-000139,m,ok
4547,BH-011,43.36,2025-05-12,HD-002439,m,ok
4548,BH-300,44.36,2023-08-18,HD-001561,m,ok


In [32]:
head_df["unit"] = convert_to_string(head_df["unit"])
head_df["quality"] = convert_to_string(head_df["quality"])

head_df["head"] = convert_to_integer(head_df["head"])
head_df["t"] = convert_to_datetime(head_df["t"])

In [33]:
head_df[head_df["head"].round(0) == -999] = np.nan

In [34]:
head_df = head_df.dropna(subset=["head"])

In [43]:
head_df.loc[head_df["unit"].isin(["cm"]), "head"] = (head_df["head"] / 100).round(0)

In [45]:
head_df.loc[head_df["unit"].isin(["cm"]), "unit"] = "m"

In [46]:
head_df.loc[head_df["unit"].isin(["cm"]), "unit"]

Series([], Name: unit, dtype: string)

In [48]:
head_df.to_parquet(f"{save_file_path}head.parquet")